# LBSTER를 이용한 임베딩 추출

이 튜토리얼에서는 사전 훈련된 LBSTER 모델을 사용하여 단백질 서열에서 임베딩을 추출하는 방법을 보여줍니다.

## 설정 및 설치

먼저 LBSTER가 설치되어 있는지 확인하십시오:

```bash
pip install -e .
```

필요한 라이브러리를 가져오는 것으로 시작하겠습니다:

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

In [ ]:
from lobster.model import LobsterPMLM, LobsterCBMPMLM

## 사전 훈련된 모델 로드

LBSTER는 여러 사전 훈련된 모델을 제공합니다. 마스크 언어 모델을 로드해 보겠습니다:

In [ ]:
# Choose a model to use
model_name = "asalam91/lobster_24M"  # 24M parameter model

In [ ]:
# Load the model
model = LobsterPMLM(model_name)
model.eval()  # Set to evaluation mode

In [ ]:
# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Model loaded on {device}")

## 샘플 단백질 서열

임베딩을 추출할 샘플 단백질 서열을 정의해 보겠습니다:

In [ ]:
sequences = [
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR",  # Hemoglobin alpha
    "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH",  # Hemoglobin beta
    "MNIFEMLRIDEGLRLKIYKDTEGYYTIGIGHLLTKSPSLNAAKSELDKAIGRNTNGVITKDEAEKLFNQDVDAAVRGILRNAKLKPVYDSLDAVRRAALINMVFQMGETGVAGFTNSLRMLQQKRWDEAAVNLAKSRWYNQTPNRAKRVITTFRTGTWDAYKNL",  # T4 Lysozyme
    "MEAPAAGAAPPPGPALGNGVAGAGGEAAAAPGGGGEAPARKRGRPGGDNHGPGREARDGPRERLGAGPADAGPGAPGSQHPGGRGRGGGPGLSTLPGGGPGPGGFGPLGFPMRGRGGPGPGGFGPRGGPGAAGFPTRGRGGGPGPDGF",  # Heterogeneous Nuclear Ribonucleoprotein A1
]

## 임베딩 추출

이제 모델을 사용하여 이러한 시퀀스에서 임베딩을 추출합니다:

In [ ]:
# Turn off gradient calculation for inference
with torch.no_grad():
    # Get embeddings for each sequence
    embeddings = []
    for seq in sequences:
        # Tokenize and process the sequence
        tokens = model.tokenizer(seq, return_tensors="pt").to(device)
        
        # Get the embedding (using the [CLS] token representation)
        outputs = model.model(
            input_ids=tokens["input_ids"],
            attention_mask=tokens["attention_mask"]
        )
        
        # Extract the [CLS] token embedding
        cls_embedding = outputs[:, 0, :].cpu().numpy()
        embeddings.append(cls_embedding.squeeze())
    
    # Convert list to numpy array
    embeddings = np.array(embeddings)

In [ ]:
print(f"Embedding shape: {embeddings.shape}")

## 임베딩 시각화

PCA를 사용하여 차원을 줄여 임베딩을 시각화해 보겠습니다:

In [ ]:
# Reduce dimensions with PCA
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

In [ ]:
# Plot the embeddings
plt.figure(figsize=(10, 8))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=100)

In [ ]:
# Add labels
for i, seq_name in enumerate(["Hemoglobin α", "Hemoglobin β", "T4 Lysozyme", "hnRNP A1"]):
    plt.annotate(seq_name, (embeddings_2d[i, 0], embeddings_2d[i, 1]), fontsize=12)

In [ ]:
plt.title("PCA of Protein Embeddings", fontsize=14)
plt.xlabel("PC1", fontsize=12)
plt.ylabel("PC2", fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 개념 병목 모델 사용

LBSTER는 해석 가능한 개념을 제공할 수 있는 개념 병목 모델도 제공합니다:

In [ ]:
# Load a concept bottleneck model
cb_model_name = "asalam91/cb_lobster_24M"
cb_model = LobsterCBMPMLM(cb_model_name)
cb_model.eval()
cb_model = cb_model.to(device)

In [ ]:
# Extract concepts
concepts = []
with torch.no_grad():
    for seq in sequences:
        # Tokenize and process the sequence
        tokens = cb_model.tokenizer(seq, return_tensors="pt").to(device)
        
        # Get concepts
        outputs = cb_model.model(
            input_ids=tokens["input_ids"],
            attention_mask=tokens["attention_mask"],
            inference=True
        )
        
        # Extract the concepts
        seq_concepts = outputs["concepts"].cpu().numpy().squeeze()
        concepts.append(seq_concepts)
    
    # Convert list to numpy array
    concepts = np.array(concepts)

In [ ]:
print(f"Concepts shape: {concepts.shape}")

## 상위 개념 분석

In [ ]:
# Display top 5 concepts for each sequence
concept_names = cb_model.concept_names[:concepts.shape[1]]  # Get the concept names

In [ ]:
for i, seq_name in enumerate(["Hemoglobin α", "Hemoglobin β", "T4 Lysozyme", "hnRNP A1"]):
    # Get the top 5 concept indices for this sequence
    top_concept_indices = np.argsort(concepts[i])[-5:][::-1]
    
    # Display the top concepts and their values
    print(f"\nTop concepts for {seq_name}:")
    for idx in top_concept_indices:
        print(f"  {concept_names[idx]}: {concepts[i][idx]:.4f}")

## 결론

이 튜토리얼에서는 다음을 시연했습니다:

1. 사전 훈련된 LBSTER 모델 로드
2. 단백질 서열에서 임베딩 추출
3. PCA를 사용하여 이러한 임베딩 시각화
4. 개념 병목 모델을 사용하여 해석 가능한 개념 추출 및 분석

이러한 임베딩은 클러스터링, 분류 또는 단백질 서열 시각화와 같은 다양한 다운스트림 작업에 사용할 수 있습니다.